In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.processing import ScriptProcessor
from sagemaker.processing import ProcessingInput
import boto3

In [ ]:
role = get_execution_role()
session = sagemaker.Session()
default_bucket = session.default_bucket()

# Subir el script 3 actualizado a S3
s3 = boto3.client('s3')
s3.upload_file('TEST_PS_PE_3_reglas_negocio.py', default_bucket, 'scripts/PE/TEST_PS_PE_3_reglas_negocio.py')
print(f'Script subido a s3://{default_bucket}/scripts/PE/TEST_PS_PE_3_reglas_negocio.py')

In [ ]:
# URIs de los outputs existentes de la corrida 1nhbm3v4v8up
base = 's3://sagemaker-us-east-2-399723489351/Pipeline-PedidoSugerido-Peru/1nhbm3v4v8up'
s3_limpieza = f'{base}/PS-Peru-Paso1-Limpieza/output/output_limpieza'
s3_modelado = f'{base}/PS-Peru-Paso2-Modelado/output/output_modelado'

print(f'Limpieza: {s3_limpieza}')
print(f'Modelado: {s3_modelado}')

In [ ]:
processor = ScriptProcessor(
    command=['python3'],
    image_uri=sagemaker.image_uris.retrieve('sklearn', session.boto_region_name, '1.2-1'),
    role=role,
    instance_count=1,
    instance_type='ml.m5.2xlarge'
)

processor.run(
    code=f's3://{default_bucket}/scripts/PE/TEST_PS_PE_3_reglas_negocio.py',
    inputs=[
        ProcessingInput(source=s3_limpieza, destination='/opt/ml/processing/input/limpieza'),
        ProcessingInput(source=s3_modelado, destination='/opt/ml/processing/input/modelado'),
    ]
)